In [ ]:
import pandas as pd
import re

def clean_llm_output(row):
    prediction = str(row['llm_prediction']).strip()
    q_type = row['type']
    
    # 1. Handle Refusals
    refusal_phrases = ["i can't", "i cannot", "is there anything else", "sorry"]
    if any(phrase in prediction.lower() for phrase in refusal_phrases):
        return "Refusal"

    # 2. Extract Ranking (A > B > C) or [A, B, C]
    if q_type == "ranking":
        # Check for bracketed list format first: [Food A, Food B]
        if '[' in prediction and ']' in prediction:
            match = re.search(r'\[(.*?)\]', prediction)
            if match:
                content = match.group(1)
                # Split by comma, remove quotes, and join with >
                items = [i.strip().replace("'", "").replace('"', "") for i in content.split(',')]
                return " > ".join(items)

        # Standard > format check
        lines = prediction.split('\n')
        for line in reversed(lines):
            if '>' in line:
                # Remove markdown bolding and quotes
                clean = re.sub(r'[*\'"]', '', line) 
                if ":" in clean:
                    clean = clean.split(":")[-1]
                return clean.strip()
        return "" 

    # 3. Extract Multiple Choice (A, B, C, or D)
    elif q_type == "multiple_choice":
        # Looks for single letters A-D surrounded by word boundaries or parentheses
        match = re.search(r'\b([A-D])\b', prediction.upper())
        return match.group(1) if match else ""

    # 4. Extract Quant. Recall (Yes/No)
    elif q_type == "quantitative_recall":
        clean_pred = prediction.lower()
        if "yes" in clean_pred[:10]: return "Yes"
        if "no" in clean_pred[:10]: return "No"
        return ""

    return prediction

# List of files to process
files = [
    "eval_results_nutrients_llama8B.csv",
    "llama31_8b_nutrient_results.csv"
]

for file_path in files:
    try:
        df = pd.read_csv(file_path)
        

        df['cleaned_prediction'] = df.apply(clean_llm_output, axis=1)
        

        df.to_csv(file_path, index=False)
        print(f"✅ Processed and saved: {file_path}")
    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")